# Demanda Aérea en Chile: Patrones y Oportunidades

Este notebook explora el tráfico aéreo histórico de Chile a partir de datos del Portal de Datos Abiertos del Gobierno de Chile, publicados por la Junta de Aeronáutica Civil (JAC). La unidad de observación corresponde a una ruta operada por una aerolínea en un mes y año; `PASAJEROS` se utiliza como indicador de demanda.

**Pregunta de investigación:** ¿Cómo varía la demanda de pasajeros según el origen, destino, aerolínea y mes, y qué patrones pueden identificarse en el tráfico aéreo de Chile?

**Objetivo.** Examinar la estructura y calidad de los datos, describir las variables relevantes, estudiar sus relaciones e identificar patrones temporales que aporten evidencia preliminar para responder la pregunta.

El notebook se organiza en cuatro secciones: (1) estructura, calidad y preparación de los datos; (2) análisis univariado; (3) relaciones entre variables; y (4) análisis temporal y contextual.


## 0. Carga y reconocimiento del dataset

In [1]:
import pandas as pd
df = pd.read_csv("trafico_historico.csv", sep=";")
df.head()

,Año,Mes,Cod_Operador,Operador,Grupo,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,ORIG_1_PAIS,...,ORIG_2_PAIS,DEST_2_PAIS,OPER_2,NAC,PAX_LIB,PASAJEROS,CAR_LIB,CARGA (Ton),CORREO,Distancia
0,1984,1,31,FAST AIR,FAST AIR,ARI,NYC,ARICA,NEW YORK,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0,0.0,13.094,0.0,7369
1,1984,1,31,FAST AIR,FAST AIR,SCL,BOG,SANTIAGO,BOGOTA,CHILE,...,CHILE,COLOMBIA,SALEN,INTERNACIONAL,0.0,0,0.0,29.086,0.0,4247
2,1984,1,31,FAST AIR,FAST AIR,SCL,MIA,SANTIAGO,MIAMI,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0,0.0,33.627,0.0,6653
3,1984,1,31,FAST AIR,FAST AIR,SCL,PTY,SANTIAGO,CIUDAD DE PANAMÁ,CHILE,...,CHILE,PANAMA,SALEN,INTERNACIONAL,0.0,0,0.0,0.000,0.0,4805
4,1984,1,31,FAST AIR,FAST AIR,SCL,SAO,SANTIAGO,SAO PAULO,CHILE,...,CHILE,BRASIL,SALEN,INTERNACIONAL,0.0,0,0.0,62.002,0.0,2585


## 1. Estructura, calidad y limpieza de los datos

### Procedencia y descripción del conjunto de datos

El archivo analizado es `trafico_historico.csv`, correspondiente al dataset **Tráfico aéreo histórico** del Portal de Datos Abiertos del Gobierno de Chile (JAC). Registra tráfico nacional e internacional asociado a Chile. De acuerdo con el alcance del proyecto, cada fila representa una ruta operada por una aerolínea en un mes y año; `PASAJEROS` es la variable principal de interés. El README declara cobertura entre 1984 y 2025; a continuación se contrasta esa descripción con el archivo disponible.


### Dimensiones del conjunto de datos

Primero revisaremos cuántos registros y variables contiene el archivo. Esta comprobación permite dimensionar el análisis y contrastar la cantidad de observaciones descrita en el README con el CSV utilizado, sin modificar los datos.

In [2]:
filas, columnas = df.shape

print(f"Cantidad de registros: {filas:,}")
print(f"Cantidad de variables: {columnas}")

Cantidad de registros: 251,117
Cantidad de variables: 25


### Variables y tipos de datos

Revisaremos los nombres originales y los tipos que pandas asignó al cargar el archivo completo. Los tipos ayudan a detectar columnas que podrían requerir atención antes de analizarlas; en esta etapa solo inspeccionaremos, sin convertirlas.

In [3]:
print("Nombres de las variables:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes.to_string())

Nombres de las variables:
['Año', 'Mes', 'Cod_Operador', 'Operador', 'Grupo', 'ORIG_1', 'DEST_1', 'ORIG_1_N', 'DEST_1_N', 'ORIG_1_PAIS', 'DEST_1_PAIS', 'ORIG_2', 'DEST_2', 'ORIG_2_N', 'DEST_2_N', 'ORIG_2_PAIS', 'DEST_2_PAIS', 'OPER_2', 'NAC', 'PAX_LIB', 'PASAJEROS', 'CAR_LIB', 'CARGA (Ton)', 'CORREO', 'Distancia']

Tipos de datos:
Año               int64
Mes               int64
Cod_Operador        str
Operador            str
Grupo               str
ORIG_1              str
DEST_1              str
ORIG_1_N            str
DEST_1_N            str
ORIG_1_PAIS         str
DEST_1_PAIS         str
ORIG_2              str
DEST_2              str
ORIG_2_N            str
DEST_2_N            str
ORIG_2_PAIS         str
DEST_2_PAIS         str
OPER_2              str
NAC                 str
PAX_LIB         float64
PASAJEROS         int64
CAR_LIB         float64
CARGA (Ton)     float64
CORREO          float64
Distancia         int64


### Cobertura temporal y geográfica

Verificaremos los años y meses presentes y el alcance geográfico usando las variables de origen, destino y país disponibles. Esto permite confirmar si el periodo descrito en el README coincide con el archivo y si hay cobertura útil para comparar rutas nacionales e internacionales.

In [4]:
if "Año" in df.columns:
    print(f"Años observados: {df['Año'].min()}–{df['Año'].max()}")
    print(f"Cantidad de años distintos: {df['Año'].nunique()}")
if "Mes" in df.columns:
    print("Meses observados:", sorted(df["Mes"].dropna().unique().tolist()))

for columna in ["NAC", "ORIG_1", "DEST_1", "ORIG_1_PAIS", "DEST_1_PAIS"]:
    if columna in df.columns:
        print(f"{columna}: {df[columna].nunique(dropna=True)} categorías distintas")

if {"ORIG_1", "DEST_1"}.issubset(df.columns):
    rutas_distintas = df[["ORIG_1", "DEST_1"]].drop_duplicates().shape[0]
    print(f"Pares origen-destino distintos: {rutas_distintas:,}")

Años observados: 1984–2025
Cantidad de años distintos: 42
Meses observados: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
NAC: 2 categorías distintas
ORIG_1: 415 categorías distintas
DEST_1: 420 categorías distintas
ORIG_1_PAIS: 78 categorías distintas
DEST_1_PAIS: 80 categorías distintas
Pares origen-destino distintos: 1,689


### Combinaciones de año y mes sin registros

Que aparezcan los 12 meses en el dataset no garantiza que cada año tenga observaciones en todos ellos. Compararemos todas las combinaciones año-mes posibles entre el primer y el último año observados con las combinaciones presentes en los datos.

In [5]:
periodo_anios = range(int(df["Año"].min()), int(df["Año"].max()) + 1)
combinaciones_esperadas = pd.MultiIndex.from_product([periodo_anios, range(1, 13)], names=["Año", "Mes"])
combinaciones_observadas = pd.MultiIndex.from_frame(df[["Año", "Mes"]].drop_duplicates())
meses_ausentes = combinaciones_esperadas.difference(combinaciones_observadas)
print(f"Combinaciones año-mes sin registros: {len(meses_ausentes):,}")
if len(meses_ausentes):
    display(meses_ausentes.to_frame(index=False))
else:
    print("No se encontraron meses ausentes dentro del periodo observado.")

Combinaciones año-mes sin registros: 0
No se encontraron meses ausentes dentro del periodo observado.


La comprobación encontró **0 combinaciones año-mes ausentes** entre 1984 y 2025: hay registros en los 504 meses del periodo observado. Esta continuidad agregada permite comparar series mensuales sin meses vacíos en el conjunto completo. No garantiza, sin embargo, que cada ruta u operador tenga observaciones en todos los meses; por ello, las comparaciones desagregadas deben considerar su propia cobertura.


### Valores faltantes

Cuantificaremos los faltantes por variable, tanto en cantidad como en porcentaje. Esto permite evaluar qué variables están completas y considerar si los faltantes afectan el análisis de demanda o variables contextuales.

In [6]:
faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
})
faltantes = faltantes.sort_values("faltantes", ascending=False)
faltantes[faltantes["faltantes"] > 0]

,faltantes,porcentaje
CORREO,57062,22.72
PAX_LIB,57062,22.72
DEST_2_PAIS,6,0.00
DEST_2_N,6,0.00
DEST_1_PAIS,4,0.00
DEST_1_N,4,0.00
ORIG_1_N,2,0.00
ORIG_1_PAIS,2,0.00


La inspección confirma **57.062 valores faltantes en `PAX_LIB` y 57.062 en `CORREO`**, equivalentes al **22,72 %** de los registros en cada variable. Esto limita los análisis que utilicen esas columnas; si se comparan grupos o periodos, las diferencias de cobertura podrían influir en los resultados. La cantidad igual no demuestra que sean exactamente las mismas filas. No se imputan valores ni se eliminan observaciones.

### Registros duplicados

Contaremos filas completamente idénticas, sin eliminarlas. Los duplicados exactos pueden inflar los conteos de pasajeros si son errores de registro; aun así, primero hay que revisar su cantidad y contexto antes de decidir qué hacer.

In [7]:
duplicados_exactos = df.duplicated().sum()
porcentaje_duplicados = (duplicados_exactos / len(df) * 100) if len(df) else 0
print(f"Filas completamente duplicadas: {duplicados_exactos:,}")
print(f"Porcentaje del dataset: {porcentaje_duplicados:.2f}%")

Filas completamente duplicadas: 18
Porcentaje del dataset: 0.01%


Para entender qué representan los duplicados, mostraremos ejemplos de las filas que aparecen más de una vez. El filtro conserva todas las copias para inspeccionarlas; no elimina registros.

In [8]:
mascara_duplicados = df.duplicated(keep=False)
ejemplos_duplicados = df.loc[mascara_duplicados].drop_duplicates().head(5)
print(f"Filas que forman parte de grupos duplicados: {mascara_duplicados.sum():,}")
display(ejemplos_duplicados)

Filas que forman parte de grupos duplicados: 36


,Año,Mes,Cod_Operador,Operador,Grupo,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,ORIG_1_PAIS,...,ORIG_2_PAIS,DEST_2_PAIS,OPER_2,NAC,PAX_LIB,PASAJEROS,CAR_LIB,CARGA (Ton),CORREO,Distancia
66104,1994,11,VA,VIASA,VIASA,SCL,NYC,SANTIAGO,NEW YORK,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0,0.0,0.000,0.0,8232
98427,1998,6,RG,VARIG,VARIG,AMS,SCL,AMSTERDAM,SANTIAGO,HOLANDA,...,CHILE,HOLANDA,LLEGAN,INTERNACIONAL,0.0,0,0.0,0.000,0.0,12000
98457,1998,6,RG,VARIG,VARIG,TYO,SCL,TOKYO,SANTIAGO,JAPON,...,CHILE,JAPON,LLEGAN,INTERNACIONAL,0.0,0,0.0,0.000,0.0,17149
101862,1998,11,RG,VARIG,VARIG,FLN,SCL,FLORIANOPOLIS,SANTIAGO,BRASIL,...,CHILE,BRASIL,LLEGAN,INTERNACIONAL,0.0,0,0.0,0.000,0.0,2480
106660,1999,7,PO,POLAR AIR CARGO,POLAR AIR CARGO,SCL,MIA,SANTIAGO,MIAMI,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0,0.0,14.936,0.0,6653


El conteo confirma **18 filas duplicadas exactas** (0,01 % al redondear a dos decimales). La tabla muestra ejemplos de filas que aparecen más de una vez; en total, 36 filas participan en esos pares. Como son idénticas en todas las columnas, podrían inflar agregaciones si representan duplicaciones de registro. Conviene investigar su procedencia antes de eliminarlas, pues el conteo por sí solo no explica cómo se generaron. Se conservan todas.

### Rangos y categorías

Comprobaremos condiciones básicas para variables cuyo dominio es interpretable —por ejemplo, meses entre 1 y 12 y pasajeros no negativos— y revisaremos etiquetas categóricas con espacios o diferencias de mayúsculas. Estas señales identifican valores para investigar; no corrigen ni descartan registros.

In [9]:
columnas_numericas = df.select_dtypes(include="number").columns
print("Rangos observados en variables numéricas:")
display(df[columnas_numericas].agg(["min", "max"]).T)

if "Mes" in df.columns:
    meses_invalidos = df.loc[~df["Mes"].between(1, 12), "Mes"]
    print(f"\nRegistros con Mes fuera de 1–12: {len(meses_invalidos):,}")
    if len(meses_invalidos):
        print("Valores observados:", sorted(meses_invalidos.dropna().unique().tolist()))

for columna in ["PASAJEROS", "Distancia"]:
    if columna in df.columns:
        cantidad_negativos = (df[columna] < 0).sum()
        print(f"Registros con {columna} negativo: {cantidad_negativos:,}")

columnas_categoricas = df.select_dtypes(include=["object", "string"]).columns
resumen_categorias = pd.DataFrame({
    "categorias_distintas": df[columnas_categoricas].nunique(dropna=True),
    "con_espacios_exteriores": pd.Series({
        col: df[col].dropna().astype(str).str.strip().ne(df[col].dropna().astype(str)).sum()
        for col in columnas_categoricas
    })
}, index=columnas_categoricas)
print("\nResumen de categorías y etiquetas con espacios exteriores:")
display(resumen_categorias.sort_values("categorias_distintas", ascending=False))

posibles_variantes = {}
for columna in columnas_categoricas:
    etiquetas = df[columna].dropna().astype(str)
    grupos = etiquetas.groupby(etiquetas.str.strip().str.upper()).unique()
    variantes = {etiqueta: valores.tolist() for etiqueta, valores in grupos.items() if len(valores) > 1}
    if variantes:
        posibles_variantes[columna] = variantes

print("\nEtiquetas distintas que coinciden al ignorar espacios exteriores y mayúsculas:")
if posibles_variantes:
    for columna, variantes in posibles_variantes.items():
        print(f"{columna}: {variantes}")
else:
    print("No se detectaron variantes con este criterio.")

print("\nValores con espacios exteriores y posibles categorías equivalentes tras quitar esos espacios:")
for columna in ["Operador", "ORIG_1_N", "DEST_2_N"]:
    if columna in df.columns:
        valores = df[columna].dropna().astype(str)
        con_espacios = valores[valores.str.strip().ne(valores)]
        print(f"\n{columna}: {len(con_espacios):,} registros afectados")
        for valor, frecuencia in con_espacios.value_counts().head(10).items():
            equivalente = valores[valores.eq(valor.strip())].shape[0]
            print(f"  valor={valor!r} | frecuencia={frecuencia:,} | frecuencia de versión sin espacios={equivalente:,}")


Rangos observados en variables numéricas:


,min,max
Año,1984.000,2.025000e+03
Mes,1.000,1.200000e+01
PAX_LIB,0.000,1.980160e+05
PASAJEROS,0.000,5.085500e+04
CAR_LIB,0.000,4.461830e+05
CARGA (Ton),-1.726,1.209088e+04
CORREO,0.000,4.148245e+09
Distancia,1.000,2.945600e+04



Registros con Mes fuera de 1–12: 0
Registros con PASAJEROS negativo: 0
Registros con Distancia negativo: 0

Resumen de categorías y etiquetas con espacios exteriores:


,categorias_distintas,con_espacios_exteriores
DEST_2,421,0
DEST_1,420,0
DEST_2_N,417,2
ORIG_1,415,0
ORIG_1_N,410,2
DEST_1_N,408,0
Cod_Operador,148,0
Operador,147,128
Grupo,132,0
DEST_2_PAIS,83,0



Etiquetas distintas que coinciden al ignorar espacios exteriores y mayúsculas:
No se detectaron variantes con este criterio.

Valores con espacios exteriores y posibles categorías equivalentes tras quitar esos espacios:

Operador: 128 registros afectados
  valor='JETSMART PERÚ ' | frecuencia=128 | frecuencia de versión sin espacios=0

ORIG_1_N: 2 registros afectados
  valor='TOCOPILLA/AD BARRILES / CHILE\xa0' | frecuencia=2 | frecuencia de versión sin espacios=0

DEST_2_N: 2 registros afectados
  valor='TOCOPILLA/AD BARRILES / CHILE\xa0' | frecuencia=2 | frecuencia de versión sin espacios=0


La inspección confirma **128 registros de `Operador`**, **2 de `ORIG_1_N`** y **2 de `DEST_2_N`** con espacios exteriores. Los valores visibles son `JETSMART PERÚ ` y `TOCOPILLA/AD BARRILES / CHILE\xa0`. En este archivo no aparece una categoría idéntica a la versión sin esos espacios (`JETSMART PERÚ` o `TOCOPILLA/AD BARRILES / CHILE`), así que no se observó una división actual entre ambas formas. Aun así, los espacios invisibles pueden dificultar filtros y coincidencias exactas; si hubiera versiones equivalentes en otros datos, también podrían fragmentar conteos por operador, origen o destino. No se normalizaron las etiquetas.

### Revisión de valores negativos en CARGA (Ton)

Aunque se espera revisar rangos, un valor negativo de carga requiere contexto. Contaremos estos registros y mostraremos algunos campos asociados para investigar el caso sin asumir que sea un error ni cambiar el dato.

In [10]:
negativos_carga = df["CARGA (Ton)"] < 0
columnas_contexto = ["Año", "Mes", "Operador", "ORIG_1", "DEST_1", "CARGA (Ton)", "CORREO", "PASAJEROS", "NAC"]
columnas_contexto = [col for col in columnas_contexto if col in df.columns]
print(f"Registros con CARGA (Ton) negativa: {negativos_carga.sum():,}")
display(df.loc[negativos_carga, columnas_contexto].head(10))

Registros con CARGA (Ton) negativa: 1


,Año,Mes,Operador,ORIG_1,DEST_1,CARGA (Ton),CORREO,PASAJEROS,NAC
245796,2024,12,LATAM CARGO,SCL,BOG,-1.726,0.0,0,INTERNACIONAL


En los datos observados, el mínimo de `CARGA (Ton)` leído por pandas es **−1.726** (valor decimal) y se encontró **1 registro negativo**. Corresponde a diciembre de 2024, LATAM CARGO, ruta SCL–BOG; el registro muestra 0 pasajeros y 0 en `CORREO`. Esto merece revisión con la fuente o definición de la variable, pero el signo no basta para concluir que sea un error. El dato se conserva.

### Revisión de valores altos en CORREO

Inspeccionaremos los registros con mayores valores de `CORREO` junto con su contexto de ruta, periodo, operador y pasajeros. La comparación ayuda a señalar valores que merecen verificación sin clasificarlos automáticamente como erróneos.

In [11]:
columnas_correo = [col for col in ["Año", "Mes", "Operador", "ORIG_1", "DEST_1", "ORIG_1_N", "DEST_1_N", "CORREO", "CARGA (Ton)", "PASAJEROS", "NAC"] if col in df.columns]
print("Diez registros con mayor valor de CORREO:")
display(df.nlargest(10, "CORREO")[columnas_correo])

Diez registros con mayor valor de CORREO:


,Año,Mes,Operador,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,CORREO,CARGA (Ton),PASAJEROS,NAC
154962,2007,10,AEROLINEAS ARGENTINAS,POA,SCL,PORTO ALEGRE,SANTIAGO,4.148245e+09,0.000,0,INTERNACIONAL
154961,2007,10,AEROLINEAS ARGENTINAS,NYC,SCL,NEW YORK,SANTIAGO,2.457561e+09,0.000,0,INTERNACIONAL
246373,2025,2,KLM,AMS,SCL,AMSTERDAM,SANTIAGO,6.355890e+05,141.503,2358,INTERNACIONAL
227264,2021,5,LATAM AIRLINES GROUP,MIA,SCL,MIAMI,SANTIAGO,6.347180e+05,1243.817,770,INTERNACIONAL
227994,2021,7,LATAM AIRLINES GROUP,MIA,SCL,MIAMI,SANTIAGO,5.620140e+05,1158.586,1720,INTERNACIONAL
228365,2021,8,LATAM AIRLINES GROUP,MIA,SCL,MIAMI,SANTIAGO,4.662740e+05,1140.533,4567,INTERNACIONAL
222808,2020,5,LATAM AIRLINES GROUP,MIA,SCL,MIAMI,SANTIAGO,4.634260e+05,1224.786,1955,INTERNACIONAL
223420,2020,7,LATAM AIRLINES GROUP,MIA,SCL,MIAMI,SANTIAGO,4.510660e+05,1157.012,1611,INTERNACIONAL
224065,2020,9,LATAM AIRLINES GROUP,MIA,SCL,MIAMI,SANTIAGO,4.376690e+05,1245.938,3646,INTERNACIONAL
224441,2020,10,LATAM AIRLINES GROUP,MIA,SCL,MIAMI,SANTIAGO,4.322070e+05,1933.732,4439,INTERNACIONAL


Los dos valores más altos de `CORREO` son **4.148.245.086** (AEROLINEAS ARGENTINAS, POA–SCL, octubre de 2007) y **2.457.561.050** (AEROLINEAS ARGENTINAS, NYC–SCL, octubre de 2007); el tercer valor es 635.589. Además, los dos registros máximos muestran 0 pasajeros. La diferencia de magnitud justifica revisar unidades, codificación o fuente, pero no demuestra que los valores sean incorrectos. No se modifican ni descartan.

### Posibles valores atípicos en PASAJEROS

Como la demanda es la variable objetivo, cuantificaremos valores que superan el umbral superior del rango intercuartílico (Q3 + 1.5 × RIC). Este criterio sirve para señalar observaciones extremas para revisión; no implica que sean errores, especialmente en datos de demanda agregados.

In [12]:
q1 = df["PASAJEROS"].quantile(0.25)
q3 = df["PASAJEROS"].quantile(0.75)
ric = q3 - q1
umbral_inferior = q1 - 1.5 * ric
umbral_superior = q3 + 1.5 * ric

mascara_atipicos_superiores = df["PASAJEROS"] > umbral_superior
print(f"Q1: {q1:,.2f} | Q3: {q3:,.2f} | RIC: {ric:,.2f}")
print(f"Umbral superior (Q3 + 1.5 × RIC): {umbral_superior:,.2f}")
print(f"Observaciones sobre el umbral superior: {mascara_atipicos_superiores.sum():,} ({mascara_atipicos_superiores.mean() * 100:.2f}%)")
print("\nResumen de PASAJEROS:")
print(df["PASAJEROS"].describe().to_string())

Q1: 7.00 | Q3: 1,342.00 | RIC: 1,335.00
Umbral superior (Q3 + 1.5 × RIC): 3,344.50
Observaciones sobre el umbral superior: 35,514 (14.14%)

Resumen de PASAJEROS:
count    251117.000000
mean       1714.640693
std        4041.933768
min           0.000000
25%           7.000000
50%         200.000000
75%        1342.000000
max       50855.000000


El criterio IQR identifica observaciones estadísticamente extremas, pero no demuestra que sean errores. Parte de la variabilidad puede corresponder a diferencias reales entre rutas, operadores y periodos. La distribución de `PASAJEROS` se estudiará con más detalle en la sección 2; por ahora no se modifica ni descarta ningún registro.

### Diagnóstico de ceros y valores negativos

Los ceros pueden representar situaciones distintas según la variable: en PASAJEROS afectan directamente la medida de demanda, mientras que en carga, correo, distancia u otras variables pueden corresponder a observaciones reales o a la forma de registro. Separar ceros, valores positivos, negativos y faltantes permite describir la calidad sin asumir que una categoría es un error.

Los porcentajes de la tabla se calculan sobre el total de filas del dataset. Las cuatro categorías son excluyentes y deben sumar todas las filas para cada variable.

In [13]:
variables_revision = [
    "PASAJEROS", "PAX_LIB", "CAR_LIB",
    "CARGA (Ton)", "CORREO", "Distancia"
]
variables_revision = [col for col in variables_revision if col in df.columns]

filas_diagnostico = []
for columna in variables_revision:
    serie = df[columna]
    if not pd.api.types.is_numeric_dtype(serie):
        raise TypeError(f"{columna} no tiene un tipo numérico; requiere revisión antes de clasificar signos.")

    conteos = {
        "Cero": int(serie.eq(0).sum()),
        "Positivo": int(serie.gt(0).sum()),
        "Negativo": int(serie.lt(0).sum()),
        "Faltante": int(serie.isna().sum())
    }
    assert sum(conteos.values()) == len(df), f"Las categorías no cubren todas las filas de {columna}."

    fila = {"Variable": columna}
    for categoria, cantidad in conteos.items():
        fila[f"{categoria} (n)"] = cantidad
        fila[f"{categoria} (%)"] = round(cantidad / len(df) * 100, 4) if len(df) else 0
    filas_diagnostico.append(fila)

diagnostico_signos = pd.DataFrame(filas_diagnostico).set_index("Variable")
display(diagnostico_signos)
print(f"Denominador de todos los porcentajes: {len(df):,} filas.")
print("En cada variable, cero + positivo + negativo + faltante suma el total de filas.")

,Cero (n),Cero (%),Positivo (n),Positivo (%),Negativo (n),Negativo (%),Faltante (n),Faltante (%)
Variable,,,,,,,,
PASAJEROS,49202,19.5933,201915,80.4067,0,0.0000,0,0.0000
PAX_LIB,158925,63.2872,35130,13.9895,0,0.0000,57062,22.7233
CAR_LIB,246589,98.1969,4528,1.8031,0,0.0000,0,0.0000
CARGA (Ton),117293,46.7085,133823,53.2911,1,0.0004,0,0.0000
CORREO,182280,72.5877,11775,4.6890,0,0.0000,57062,22.7233
Distancia,0,0.0000,251117,100.0000,0,0.0000,0,0.0000


Denominador de todos los porcentajes: 251,117 filas.
En cada variable, cero + positivo + negativo + faltante suma el total de filas.


La tabla confirma que PASAJEROS no contiene valores negativos ni faltantes; los ceros representan cerca de una quinta parte de los registros. PAX_LIB tiene 63,29 % de ceros y 22,72 % de faltantes; CORREO, 72,59 % de ceros y 22,72 % de faltantes. CAR_LIB es cero en 98,20 % de las filas. CARGA (Ton) tiene 46,71 % de ceros, 53,29 % de positivos y un negativo (0,0004 %); Distancia es positiva en todos los registros. Estas proporciones describen el archivo, pero no determinan por sí solas qué variables deben conservarse en los análisis principales.

### Posible filtrado de registros con pocos pasajeros

No aplicaremos un umbral mínimo ni excluiremos los registros con pocos pasajeros o con cero. Filtrarlos podría facilitar el estudio de rutas de mayor volumen, pero también ocultar rutas de baja demanda, variación estacional u operaciones alternativas. Sin conocer el significado de los ceros y valores bajos, excluirlos podría sesgar las comparaciones. La decisión queda pendiente para después de estudiar la distribución de PASAJEROS en la sección 2.

### Síntesis de estructura y calidad antes de la limpieza

El archivo original contiene **251.117 filas y 25 columnas**, con observaciones entre **1984 y 2025** y registros en las 504 combinaciones posibles de año y mes. La inspección de tipos, variables y cobertura confirma que el archivo permite estudiar diferencias de demanda por periodo, ruta, operador y tipo de tráfico, sin garantizar cobertura completa para cada ruta u operador.

Entre los problemas confirmados se encuentran **57.062 valores faltantes** en cada una de `PAX_LIB` y `CORREO` (**22,72 %**), **18 filas duplicadas exactas**, espacios exteriores en etiquetas de `Operador`, `ORIG_1_N` y `DEST_2_N`, un valor negativo en `CARGA (Ton)` y valores de `CORREO` de magnitud inusual. La regla IQR también marca **35.514 observaciones de `PASAJEROS` (14,14 %)** por sobre el umbral superior. Son señales para revisar: el criterio estadístico y las magnitudes observadas no demuestran por sí solos que los registros sean errores.

La limpieza se documenta a continuación como un proceso separado. Los valores y filas se conservaron durante esta inspección inicial; las decisiones posteriores se justifican según la pregunta del proyecto y las limitaciones verificables de los datos.


### Limpieza y preparación del dataset

La limpieza se realizará sobre una copia independiente del dataset, conservando intactos el archivo original y el DataFrame usado en la exploración. Se recortarán espacios externos en columnas de texto, se excluirán PAX_LIB, CAR_LIB y CORREO por no ser centrales para la pregunta principal y se eliminarán, en pasos separados, las filas con nulos restantes y los duplicados exactos.

Los seis registros de AEROVIAS DAP asociados a AWC tienen información incompleta en nombres o países geográficos. Como no se pudo verificar qué representa AWC ni una corrección fiable, no se reemplazará el código: esos registros se contabilizarán y excluirán por los valores nulos que permanecen tras retirar las tres columnas. Los duplicados se identificarán únicamente cuando todas las columnas conservadas coincidan después del recorte de espacios.

In [ ]:
from pathlib import Path

ruta_original_limpieza = Path("trafico_historico.csv")
ruta_salida_limpieza = Path("trafico_limpio.csv")

# 1. Cargar el archivo original y registrar sus dimensiones.
df_original_limpieza = pd.read_csv(ruta_original_limpieza, sep=";")
filas_originales_limpieza, columnas_originales_limpieza = df_original_limpieza.shape
print(f"Dimensiones originales: {filas_originales_limpieza:,} filas × {columnas_originales_limpieza} columnas")

# 2. Trabajar sobre una copia; el DataFrame df de la exploración no se modifica.
df_limpio = df_original_limpieza.copy()

# 3. Recortar espacios externos de campos de texto.
columnas_texto_limpieza = df_limpio.select_dtypes(include=["object", "string"]).columns
for columna in columnas_texto_limpieza:
    df_limpio[columna] = df_limpio[columna].str.strip()

# 4. Excluir variables que no son centrales para el análisis principal.
columnas_excluir_limpieza = ["PAX_LIB", "CAR_LIB", "CORREO"]
columnas_eliminadas_limpieza = [
    columna for columna in columnas_excluir_limpieza
    if columna in df_limpio.columns
]
df_limpio = df_limpio.drop(columns=columnas_eliminadas_limpieza)

# 5. Contar y eliminar filas con nulos en las columnas que se conservarán.
mascara_nulos_limpieza = df_limpio.isna().any(axis=1)
filas_nulas_eliminadas_limpieza = int(mascara_nulos_limpieza.sum())
df_limpio = df_limpio.loc[~mascara_nulos_limpieza].copy()

# 6. Eliminar únicamente filas exactamente duplicadas entre las columnas restantes.
filas_antes_duplicados_limpieza = len(df_limpio)
df_limpio = df_limpio.drop_duplicates().copy()
duplicados_eliminados_limpieza = filas_antes_duplicados_limpieza - len(df_limpio)

# 7. Guardar y recargar el archivo para validar el resultado escrito.
df_limpio.to_csv(ruta_salida_limpieza, sep=";", index=False, encoding="utf-8")
df_limpio_verificacion = pd.read_csv(ruta_salida_limpieza, sep=";")

nulos_restantes_limpieza = int(df_limpio_verificacion.isna().sum().sum())
duplicados_restantes_limpieza = int(df_limpio_verificacion.duplicated().sum())
columnas_excluidas_restantes_limpieza = [
    columna for columna in columnas_excluir_limpieza
    if columna in df_limpio_verificacion.columns
]
validacion_limpieza_correcta = (
    nulos_restantes_limpieza == 0
    and duplicados_restantes_limpieza == 0
    and len(columnas_excluidas_restantes_limpieza) == 0
    and df_limpio.shape == df_limpio_verificacion.shape
)

resumen_limpieza = pd.Series({
    "Filas originales": filas_originales_limpieza,
    "Columnas originales": columnas_originales_limpieza,
    "Columnas eliminadas": len(columnas_eliminadas_limpieza),
    "Filas eliminadas por nulos": filas_nulas_eliminadas_limpieza,
    "Filas eliminadas por duplicados exactos": duplicados_eliminados_limpieza,
    "Filas finales": df_limpio_verificacion.shape[0],
    "Columnas finales": df_limpio_verificacion.shape[1],
    "Valores nulos restantes": nulos_restantes_limpieza,
    "Duplicados exactos restantes": duplicados_restantes_limpieza
}, name="Resultado")
display(resumen_limpieza.to_frame())

print("Columnas excluidas presentes en el archivo final:", columnas_excluidas_restantes_limpieza)

if validacion_limpieza_correcta:
    print("VALIDACIÓN CORRECTA: el archivo recargado no contiene nulos, duplicados exactos ni las columnas excluidas.")
else:
    print("AVISO: falló al menos una validación; revisar los resultados antes de usar el archivo.")
    print("Nulos restantes:", nulos_restantes_limpieza)
    print("Duplicados exactos restantes:", duplicados_restantes_limpieza)
    print("Columnas excluidas presentes:", columnas_excluidas_restantes_limpieza)
    print("Dimensiones coinciden:", df_limpio.shape == df_limpio_verificacion.shape)

Dimensiones originales: 251,117 filas × 25 columnas


La ejecución conserva el CSV original y genera un archivo independiente. Se excluyeron **3 columnas**, se eliminaron **6 filas por valores nulos** y, después, **18 filas por duplicados exactos**. El archivo recargado tiene **251.093 filas y 22 columnas**; las validaciones confirman **0 nulos, 0 duplicados exactos y ausencia de PAX_LIB, CAR_LIB y CORREO**.

Los seis registros retirados por nulos corresponden a AEROVIAS DAP e incluyen AWC en los campos de origen o destino, con nombres o países incompletos. No se imputó información ni se sustituyó AWC. Los duplicados se contaron después de retirar los nulos y no se eliminaron filas por similitud o por compartir ruta, mes u operador. El recorte se limitó a espacios externos, incluidos espacios Unicode, sin alterar códigos, mayúsculas ni nombres. No se aplicaron filtros adicionales.

## 2. Análisis univariado

### 2.1. Distribución de PASAJEROS

PASAJEROS es la variable objetivo del proyecto. Primero resumiremos su centro, dispersión, forma y rango para entender cómo se distribuye la demanda observada en los registros.

### Estadísticas descriptivas

Calcularemos las medidas solicitadas sobre el dataset limpio. Q2 corresponde a la mediana, por lo que se presenta en una sola fila para evitar repetir el mismo estadístico.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Reutilizar el DataFrame creado en la sección de limpieza.
# Si se ejecuta esta subsección por separado, cargar el archivo limpio.
if "df_limpio" not in globals():
    df_limpio = pd.read_csv("trafico_limpio.csv", sep=";")
    print("Dataset limpio cargado desde trafico_limpio.csv.")
else:
    print("Se reutiliza df_limpio:", df_limpio.shape)

pasajeros = df_limpio["PASAJEROS"]

def formato_numero_es(valor, decimales=2):
    texto = f"{valor:,.{decimales}f}"
    return texto.replace(",", "_").replace(".", ",").replace("_", ".")

cuartil_1 = pasajeros.quantile(0.25)
cuartil_2 = pasajeros.quantile(0.50)
cuartil_3 = pasajeros.quantile(0.75)
modas_pasajeros = pasajeros.mode()
moda_pasajeros = ", ".join(
    f"{int(valor):,}" if float(valor).is_integer() else f"{valor:.2f}"
    for valor in modas_pasajeros
)

estadisticas_pasajeros = [
    ("Observaciones", int(pasajeros.count())),
    ("Media", float(pasajeros.mean())),
    ("Mediana / Q2", float(cuartil_2)),
    ("Moda", moda_pasajeros),
    ("Desviación estándar", float(pasajeros.std())),
    ("Varianza", float(pasajeros.var())),
    ("Mínimo", float(pasajeros.min())),
    ("Q1 / percentil 25", float(cuartil_1)),
    ("Q3 / percentil 75", float(cuartil_3)),
    ("Rango intercuartílico (IQR)", float(cuartil_3 - cuartil_1)),
    ("Máximo", float(pasajeros.max())),
    ("Asimetría", float(pasajeros.skew()))
]

tabla_estadisticas_pasajeros = pd.DataFrame(
    estadisticas_pasajeros, columns=["Estadístico", "Valor"]
)
tabla_estadisticas_pasajeros["Valor"] = [
    (
        formato_numero_es(valor, 0) if nombre == "Observaciones"
        else valor if nombre == "Moda"
        else formato_numero_es(float(valor), 2)
    )
    for nombre, valor in estadisticas_pasajeros
]
display(tabla_estadisticas_pasajeros)

La distribución de `PASAJEROS` presenta una media de **1.714,80**, muy superior a la mediana de **200**, y una asimetría positiva de **4,33**. Esto indica una cola hacia registros con cantidades altas. El 50 % central va de **7 a 1.343 pasajeros** (IQR **1.336**), mientras que el rango completo va de **0 a 50.855**; la desviación estándar de **4.042,09** confirma una dispersión amplia. En el contexto del proyecto, hay observaciones con volúmenes muy diferentes, pero estas medidas por sí solas no muestran qué operadores, rutas o meses explican esa variación.

### Frecuencia de registros con cero pasajeros

Separaremos las observaciones con cero pasajeros de aquellas con un valor positivo. Esta proporción ayuda a entender la composición de la variable objetivo; los ceros se conservarán y no se tratarán automáticamente como errores.

In [ ]:
total_pasajeros = len(pasajeros)
cantidad_ceros_pasajeros = int(pasajeros.eq(0).sum())
cantidad_positivos_pasajeros = int(pasajeros.gt(0).sum())

assert cantidad_ceros_pasajeros + cantidad_positivos_pasajeros == total_pasajeros, (
    "La tabla considera solo ceros y positivos; revisar si existen negativos o faltantes."
)

tabla_frecuencia_pasajeros = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Registros con cero pasajeros",
        "Registros con pasajeros positivos"
    ],
    "Cantidad": [
        total_pasajeros,
        cantidad_ceros_pasajeros,
        cantidad_positivos_pasajeros
    ],
    "Porcentaje del total": [
        100.00,
        cantidad_ceros_pasajeros / total_pasajeros * 100,
        cantidad_positivos_pasajeros / total_pasajeros * 100
    ]
})
display(tabla_frecuencia_pasajeros.round(2))

En el dataset limpio, **49.194 registros (19,59 %)** tienen cero pasajeros y **201.899 (80,41 %)** tienen pasajeros positivos. La tabla cubre todas las observaciones; la limpieza dejó esta variable sin faltantes ni valores negativos. Los ceros son una parte sustantiva de los datos y deben mantenerse en el análisis mientras no exista evidencia para clasificarlos como errores u operaciones inválidas.

### Histograma en la escala original

El histograma permite observar cuántos registros se concentran en distintos rangos de pasajeros. Se muestra primero en la escala original, sin transformar ni excluir observaciones.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=df_limpio, x="PASAJEROS", bins=50, color="steelblue", ax=ax)
ax.set_title("Distribución de pasajeros por registro")
ax.set_xlabel("Cantidad de pasajeros")
ax.set_ylabel("Frecuencia")
fig.tight_layout()
plt.show()

En la escala original se observa una fuerte concentración en los rangos bajos y una cola prolongada hacia valores altos. La extensión del eje hasta 50.855 hace que las diferencias en los rangos bajos sean difíciles de distinguir. Por eso se incluye a continuación una vista complementaria con transformación log1p; los datos originales permanecen intactos.

### Histograma complementario con transformación log1p

Dada la asimetría elevada y la presencia de ceros, usaremos log1p, que calcula el logaritmo natural de 1 + pasajeros. Esta transformación permite visualizar mejor los rangos bajos y medios junto con la cola alta; el eje representa valores transformados, no pasajeros en su escala original.

In [ ]:
pasajeros_log1p = np.log1p(pasajeros)

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(x=pasajeros_log1p, bins=50, color="darkcyan", ax=ax)
ax.set_title("Distribución de pasajeros en escala log1p")
ax.set_xlabel("log(1 + cantidad de pasajeros)")
ax.set_ylabel("Frecuencia")
fig.tight_layout()
plt.show()

La transformación log1p separa mejor los rangos bajos y medios que quedaban comprimidos en el histograma original. Los ceros se mantienen como cero transformado, y la cola de valores altos continúa visible. Esta figura complementa la escala original: no cambia la variable usada en el dataset ni sustituye su interpretación en unidades de pasajeros.

### Diagrama de caja

El boxplot resume la mediana, los cuartiles y el rango intercuartílico. Los puntos que aparecen más allá de los bigotes son posibles valores atípicos según la regla de 1,5 × IQR; no son errores confirmados y no se eliminarán.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    x=pasajeros,
    color="lightsteelblue",
    flierprops={
        "marker": "o",
        "markersize": 1.2,
        "markerfacecolor": "darkorange",
        "markeredgecolor": "darkorange",
        "alpha": 0.18
    },
    ax=ax
)
ax.set_title("Diagrama de caja de pasajeros por registro")
ax.set_xlabel("Cantidad de pasajeros")
ax.set_ylabel("")
fig.tight_layout()
umbral_superior_boxplot = cuartil_3 + 1.5 * (cuartil_3 - cuartil_1)
cantidad_extremos_superiores = int(pasajeros.gt(umbral_superior_boxplot).sum())
print(
    f"Umbral superior del boxplot: {formato_numero_es(umbral_superior_boxplot, 0)}; "
    f"registros sobre el umbral: {formato_numero_es(cantidad_extremos_superiores, 0)} "
    f"({formato_numero_es(cantidad_extremos_superiores / len(pasajeros) * 100, 2)} %)."
)
plt.show()

La mediana del boxplot es **200 pasajeros** y la caja abarca de **7 (Q1) a 1.343 (Q3)**. El umbral superior de la regla de **1,5 × IQR** es **3.347 pasajeros**; **35.478 registros (14,13 %)** superan ese valor. Son posibles valores atípicos según este criterio estadístico, no errores confirmados: las diferencias entre rutas y operadores pueden producir cantidades muy distintas. Por eso no se eliminan y se requiere contexto para investigarlos.

El boxplot en escala original concentra la caja y los bigotes en la parte izquierda, mientras que los posibles extremos se extienden lejos. Para facilitar la lectura del tramo central, el siguiente gráfico usa log1p solo para la visualización; conserva también todos los registros.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    x=np.log1p(pasajeros),
    color="lightseagreen",
    flierprops={
        "marker": "o",
        "markersize": 1.2,
        "markerfacecolor": "darkorange",
        "markeredgecolor": "darkorange",
        "alpha": 0.18
    },
    ax=ax
)
ax.set_title("Diagrama de caja de pasajeros en escala log1p")
ax.set_xlabel("log(1 + cantidad de pasajeros)")
ax.set_ylabel("")
fig.tight_layout()
plt.show()

La escala log1p hace más visible la posición relativa de la mediana y los cuartiles, y mantiene visible la cola de valores altos. Los puntos siguen siendo observaciones potencialmente extremas según el boxplot transformado; no indican errores ni se descartan. Para interpretar las cantidades en pasajeros, se debe consultar el boxplot y las estadísticas de la escala original.

### 2.2. Frecuencia de aerolíneas

Contaremos los registros asociados a cada valor de Operador y mostraremos los diez más frecuentes. Esto describe la presencia de cada operador en el dataset; no equivale a comparar la cantidad de pasajeros transportados.

In [ ]:
frecuencia_operadores = (
    df_limpio["Operador"]
    .value_counts()
    .head(10)
    .rename_axis("Operador")
    .reset_index(name="Registros")
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=frecuencia_operadores,
    x="Registros",
    y="Operador",
    order=frecuencia_operadores["Operador"],
    color="steelblue",
    ax=ax
)
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3)
ax.set_title("Diez operadores con más registros")
ax.set_xlabel("Cantidad de registros")
ax.set_ylabel("Operador")
ax.set_xlim(0, frecuencia_operadores["Registros"].max() * 1.12)
fig.tight_layout()
plt.show()

LATAM AIRLINES GROUP presenta más registros (**51.014**), seguido de LADECO (**16.807**) y SKY (**16.594**). Esto describe la frecuencia de aparición en el dataset, no la cantidad de pasajeros transportados. La frecuencia de registros sirve para conocer la representación de cada operador, pero no permite ordenar su demanda ni inferir su tamaño o eficiencia.

### 2.3. Frecuencia de rutas

Usaremos ORIG_1 y DEST_1 como una pareja direccional del tramo. No mezclaremos las columnas _1 y _2 porque representan otra codificación del mismo registro y combinarlas podría duplicar o cambiar la definición de la ruta. El origen y el destino se conservarán en su orden, de modo que SCL → MIA y MIA → SCL sean rutas distintas. Contaremos registros, no pasajeros.

In [ ]:
frecuencia_rutas = (
    df_limpio.groupby(["ORIG_1", "DEST_1"])
    .size()
    .rename("Registros")
    .reset_index()
)
frecuencia_rutas["Ruta direccional"] = (
    frecuencia_rutas["ORIG_1"] + " → " + frecuencia_rutas["DEST_1"]
)
frecuencia_rutas = (
    frecuencia_rutas
    .sort_values(
        ["Registros", "ORIG_1", "DEST_1"],
        ascending=[False, True, True]
    )
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=frecuencia_rutas,
    x="Registros",
    y="Ruta direccional",
    order=frecuencia_rutas["Ruta direccional"],
    color="slateblue",
    ax=ax
)
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3)
ax.set_title("Diez rutas direccionales con más registros")
ax.set_xlabel("Cantidad de registros")
ax.set_ylabel("Ruta (origen → destino)")
ax.set_xlim(0, frecuencia_rutas["Registros"].max() * 1.12)
fig.tight_layout()
plt.show()

SCL → MIA es la ruta direccional con más registros (**5.336**), seguida de MIA → SCL (**5.297**). Esta frecuencia cuenta observaciones por tramo; no equivale a sumar pasajeros. La comparación de volumen de pasajeros por ruta se realiza en la sección 3.

### 2.4. Distribución de distancias

La distancia describe la longitud registrada de los tramos. Resumiremos su centro y dispersión y luego observaremos su distribución con un histograma. No compararemos distancia con pasajeros en esta sección.

In [ ]:
distancia = df_limpio["Distancia"]
cuartil_1_distancia = distancia.quantile(0.25)
mediana_distancia = distancia.quantile(0.50)
cuartil_3_distancia = distancia.quantile(0.75)

estadisticas_distancia = pd.DataFrame([
    ("Observaciones", int(distancia.count())),
    ("Media", float(distancia.mean())),
    ("Mediana / Q2", float(mediana_distancia)),
    ("Desviación estándar", float(distancia.std())),
    ("Mínimo", float(distancia.min())),
    ("Q1 / percentil 25", float(cuartil_1_distancia)),
    ("Q3 / percentil 75", float(cuartil_3_distancia)),
    ("Máximo", float(distancia.max()))
], columns=["Estadístico", "Valor"])

estadisticas_distancia["Valor"] = [
    formato_numero_es(float(valor), 0 if nombre == "Observaciones" else 2)
    for nombre, valor in estadisticas_distancia[["Estadístico", "Valor"]].itertuples(index=False, name=None)
]
display(estadisticas_distancia)

Las distancias registradas van de **1 a 29.456**, con media **3.825,50**, mediana **2.180** y un 50 % central entre **863 y 6.385**. La media superior a la mediana y la extensión hacia valores altos son compatibles con una distribución asimétrica. La unidad de `Distancia` no se confirma aquí, por lo que se mantiene la etiqueta neutral «Distancia registrada». Esta descripción no explica todavía su relación con pasajeros.

### Histograma de distancias

El histograma muestra la frecuencia de los rangos de distancia y permite complementar las estadísticas descriptivas sin relacionar todavía esta variable con PASAJEROS.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=df_limpio, x="Distancia", bins=50, color="seagreen", ax=ax)
ax.set_title("Distribución de distancias de los tramos")
ax.set_xlabel("Distancia registrada")
ax.set_ylabel("Frecuencia")
fig.tight_layout()
plt.show()

El histograma concentra más registros en distancias bajas y medias, con una cola hacia distancias mayores. Esto concuerda con la media superior a la mediana y el máximo de 29.456. La distribución describe los tramos del dataset; la relación entre distancia y cantidad de pasajeros se reservará para el punto 3.

## 3. Relaciones entre variables

En esta sección exploraremos asociaciones entre `PASAJEROS` y variables relevantes para la pregunta del proyecto. Según la unidad de observación descrita en el README, cada fila representa una ruta operada por una aerolínea en un mes y año. Por ello, “pasajeros por registro” se refiere a esa observación agregada y no a pasajeros por vuelo individual. Los resultados son descriptivos y sirven como evidencia preliminar: no permiten establecer causalidad ni explicar por sí solos las razones de las diferencias observadas.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import plotly.express as px

# Reutilizar el dataset limpio de la sección anterior o cargarlo si se ejecuta este punto por separado.
if "df_limpio" not in globals():
    df_limpio = pd.read_csv("trafico_limpio.csv", sep=";")
    print("Dataset limpio cargado desde trafico_limpio.csv:", df_limpio.shape)
else:
    print("Se reutiliza df_limpio:", df_limpio.shape)

columnas_agrupacion = ["Operador", "ORIG_1", "DEST_1", "NAC", "PASAJEROS"]
if df_limpio[columnas_agrupacion].isna().any().any():
    raise ValueError("Hay valores faltantes en las variables usadas para agrupar; revisar antes de interpretar los resultados.")
print("Comprobación de agrupaciones: las variables categóricas y PASAJEROS no tienen valores faltantes.")

def formato_numero_es(valor, decimales=0):
    texto = f"{valor:,.{decimales}f}"
    return texto.replace(",", "_").replace(".", ",").replace("_", ".")


### 3.1. Pasajeros por aerolínea

La pregunta es cómo varía la cantidad de pasajeros entre operadores. Primero sumaremos `PASAJEROS` por `Operador` para identificar los mayores acumulados del período. El total combina volumen de registros y pasajeros observados; por eso, además compararemos la distribución de pasajeros por registro entre esos diez operadores.


In [ ]:
# Agrupar por operador y ordenar según pasajeros acumulados.
pasajeros_por_operador = (
    df_limpio.groupby("Operador", dropna=True)["PASAJEROS"]
    .sum()
    .sort_values(ascending=False)
    .rename("Pasajeros acumulados")
    .reset_index()
)
top_operadores = pasajeros_por_operador.head(10).copy()
tabla_top_operadores = top_operadores.copy()
tabla_top_operadores["Pasajeros acumulados"] = tabla_top_operadores["Pasajeros acumulados"].map(formato_numero_es)
display(tabla_top_operadores)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=top_operadores,
    x="Pasajeros acumulados",
    y="Operador",
    order=top_operadores["Operador"],
    color="steelblue",
    ax=ax
)
ax.bar_label(
    ax.containers[0],
    labels=[formato_numero_es(v) for v in top_operadores["Pasajeros acumulados"]],
    padding=3
)
ax.set_title("Diez operadores con más pasajeros acumulados")
ax.set_xlabel("Pasajeros acumulados (millones en el eje; etiquetas con total)")
ax.xaxis.set_major_formatter(FuncFormatter(lambda valor, posicion: f"{formato_numero_es(valor / 1_000_000, 0)} M"))
ax.set_ylabel("Operador")
ax.set_xlim(0, top_operadores["Pasajeros acumulados"].max() * 1.15)
fig.tight_layout()
plt.show()


**Hallazgo.** LATAM AIRLINES GROUP acumula **157.909.353 pasajeros** en el período **1984–2025**, seguido por LATAM AIRLINES CHILE (**81.116.183**) y SKY (**57.655.726**).

**Interpretación.** El acumulado resume el volumen registrado en todo el período y depende también de cuántos registros aporta cada operador.

**Relación con la pregunta.** Muestra que el volumen de pasajeros varía entre operadores, una de las dimensiones de interés del proyecto.

**Limitación.** No controla años activos, cantidad de registros ni eventuales cambios de estructura o denominación de las empresas; no permite concluir eficiencia, ocupación o rentabilidad.

Para comparar pasajeros por registro, restringiremos la visualización a los diez operadores definidos por su total acumulado. Se usa `log1p` solo en el eje del boxplot: comprime la cola alta y admite los registros con cero pasajeros. Los datos originales no se modifican y cada punto de la distribución sigue representando un registro.


In [ ]:
# Conservar todas las filas de los diez operadores y transformar solo el eje de visualización.
datos_top_operadores = df_limpio[df_limpio["Operador"].isin(top_operadores["Operador"])].copy()
datos_top_operadores["PASAJEROS_log1p"] = np.log1p(datos_top_operadores["PASAJEROS"])
orden_operadores = top_operadores["Operador"].tolist()

fig, ax = plt.subplots(figsize=(11, 7))
sns.boxplot(
    data=datos_top_operadores,
    x="PASAJEROS_log1p",
    y="Operador",
    order=orden_operadores,
    hue="Operador",
    hue_order=orden_operadores,
    palette="colorblind",
    legend=False,
    flierprops={"markersize": 1.0, "alpha": 0.12},
    ax=ax
)
ax.set_title("Distribución de pasajeros por registro entre los 10 operadores principales")
ax.set_xlabel("log(1 + pasajeros por registro)")
ax.set_ylabel("Operador")
fig.tight_layout()
plt.show()

resumen_top_operadores = (
    datos_top_operadores.groupby("Operador", dropna=True)["PASAJEROS"]
    .agg(Registros="count", Mediana="median", Media="mean")
    .reindex(orden_operadores)
)
display(resumen_top_operadores.round(2))


**Hallazgo.** Entre los diez operadores seleccionados por pasajeros acumulados, las medianas por registro van de **7** pasajeros para AEROLINEAS ARGENTINAS a **3.838** para AMERICAN AIRLINES.

**Interpretación.** El centro de las distribuciones difiere entre operadores y no sigue necesariamente el orden de los totales acumulados. El boxplot usa `log1p` solo en el eje para mostrar mejor las diferencias con asimetría y ceros.

**Relación con la pregunta.** Esta comparación complementa el volumen total con una descripción de los pasajeros en cada observación agregada.

**Limitación.** La unidad es el registro agregado por ruta, operador, mes y año, no un vuelo individual. La escala transformada tampoco representa directamente diferencias en unidades originales.

### 3.2. Pasajeros por ruta

Agruparemos por `ORIG_1` y `DEST_1`, que definen el tramo en una representación direccional consistente. No mezclaremos los campos `_1` y `_2`, ni sumaremos silenciosamente los sentidos opuestos. El treemap usa el área para representar pasajeros acumulados por ruta y permite consultar los valores al pasar el cursor.


In [ ]:
# Agrupar por tramo direccional sin convertir valores faltantes en categorías.
pasajeros_por_ruta = (
    df_limpio.groupby(["ORIG_1", "DEST_1"], dropna=True)["PASAJEROS"]
    .sum()
    .rename("Pasajeros acumulados")
    .reset_index()
)
pasajeros_por_ruta["Ruta"] = (
    pasajeros_por_ruta["ORIG_1"] + " → " + pasajeros_por_ruta["DEST_1"]
)
top_rutas = (
    pasajeros_por_ruta.sort_values(
        "Pasajeros acumulados", ascending=False
    ).head(10).copy()
)
tabla_top_rutas = top_rutas[["Ruta", "Pasajeros acumulados"]].copy()
tabla_top_rutas["Pasajeros acumulados"] = tabla_top_rutas["Pasajeros acumulados"].map(formato_numero_es)
display(tabla_top_rutas)

fig = px.treemap(
    top_rutas,
    path=["Ruta"],
    values="Pasajeros acumulados",
    hover_data={
        "ORIG_1": True,
        "DEST_1": True,
        "Pasajeros acumulados": ":,d"
    },
    title="Diez rutas direccionales con más pasajeros acumulados"
)
fig.update_traces(
    texttemplate="%{label}<br>%{value:,.0f}",
    hovertemplate="<b>%{label}</b><br>Pasajeros acumulados: %{value:,.0f}<extra></extra>"
)
fig.update_layout(width=950, height=600, margin=dict(t=65, l=10, r=10, b=10))
display(fig)


**Hallazgo.** Los mayores acumulados corresponden a **ANF → SCL** (**15.718.421**), **SCL → ANF** (**15.465.335**) y **BUE → SCL** (**15.331.801**) pasajeros.

**Interpretación.** El treemap muestra los corredores direccionales con mayor volumen acumulado; origen y destino se mantienen separados mediante `ORIG_1` y `DEST_1`.

**Relación con la pregunta.** Estos corredores permiten identificar dónde se concentra más tráfico de pasajeros dentro de los registros históricos.

**Limitación.** Los totales cubren **1984–2025** y no implican mayor demanda en cada mes, una tendencia creciente ni una oportunidad comercial. No se infieren causas.

### 3.3. Comparación entre tráfico nacional e internacional

Compararemos `PASAJEROS` por registro entre las categorías de `NAC`. La tabla resume centro, dispersión y rango; añadimos también la frecuencia de ceros porque estos son numerosos en el dataset. Los boxplots muestran primero los valores originales y luego una vista complementaria con `log1p` para hacer más legible la comparación bajo la asimetría.


In [ ]:
# Resumen por categoría; los faltantes no se agrupan como una categoría adicional.
estadisticas_nac = (
    df_limpio.groupby("NAC", dropna=True)["PASAJEROS"]
    .agg(
        Registros="count",
        Media="mean",
        Mediana="median",
        Desviación_estándar="std",
        Mínimo="min",
        Q1=lambda serie: serie.quantile(0.25),
        Q3=lambda serie: serie.quantile(0.75),
        Máximo="max"
    )
)
ceros_nac = (
    df_limpio.assign(_cero=df_limpio["PASAJEROS"].eq(0))
    .groupby("NAC", dropna=True)["_cero"]
    .agg(Ceros="sum", Porcentaje_ceros=lambda serie: serie.mean() * 100)
)
estadisticas_nac = estadisticas_nac.rename(columns={"Desviación_estándar": "Desviación estándar"})
estadisticas_nac = estadisticas_nac.join(ceros_nac)
display(estadisticas_nac.round(2))

orden_nac = [categoria for categoria in ["NACIONAL", "INTERNACIONAL"] if categoria in df_limpio["NAC"].dropna().unique()]

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df_limpio,
    x="NAC",
    order=orden_nac,
    y="PASAJEROS",
    hue="NAC",
    hue_order=orden_nac,
    palette="Set2",
    legend=False,
    flierprops={"markersize": 1.0, "alpha": 0.12},
    ax=ax
)
ax.set_title("Pasajeros por registro según tipo de tráfico")
ax.set_xlabel("Tipo de tráfico")
ax.set_ylabel("Pasajeros por registro")
fig.tight_layout()
plt.show()


**Hallazgo.** La mediana es **734 pasajeros por registro** en el tráfico nacional y **76** en el internacional. La desviación estándar es **5.137,34** y **3.202,08**, respectivamente. Los ceros representan **1,71 %** de los registros nacionales y **28,86 %** de los internacionales.

**Interpretación.** La distribución nacional tiene un centro más alto y mayor dispersión absoluta; en la internacional, una proporción mayor de registros tiene cero pasajeros. Ambas distribuciones presentan variabilidad amplia.

**Relación con la pregunta.** La comparación caracteriza diferencias de demanda registradas entre los dos tipos de tráfico.

**Limitación.** Son pasajeros por observación agregada, no por viajero único ni necesariamente por vuelo completo. El análisis no identifica las causas de las diferencias.

La escala original permite leer cantidades directamente, pero la asimetría puede comprimir las cajas. La siguiente vista transforma solo el eje vertical con `log1p`; incluye ceros y todos los registros, y complementa —no reemplaza— el boxplot original.


In [ ]:
datos_nac_log = df_limpio[["NAC", "PASAJEROS"]].copy()
datos_nac_log["PASAJEROS_log1p"] = np.log1p(datos_nac_log["PASAJEROS"])
orden_nac = [categoria for categoria in ["NACIONAL", "INTERNACIONAL"] if categoria in datos_nac_log["NAC"].unique()]

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=datos_nac_log,
    x="NAC",
    y="PASAJEROS_log1p",
    order=orden_nac,
    hue="NAC",
    hue_order=orden_nac,
    palette="Set2",
    legend=False,
    flierprops={"markersize": 1.0, "alpha": 0.12},
    ax=ax
)
ax.set_title("Comparación complementaria en escala log1p")
ax.set_xlabel("Tipo de tráfico")
ax.set_ylabel("log(1 + pasajeros por registro)")
fig.tight_layout()
plt.show()


### 3.4. Relación entre distancia y pasajeros

`Distancia` y `PASAJEROS` son variables cuantitativas pertinentes para explorar una asociación entre la longitud del tramo y el volumen registrado. Calcularemos estadísticas descriptivas sobre todos los datos y usaremos una muestra aleatoria reproducible solo para el gráfico, de modo que la nube de puntos sea legible. Se conservarán los ceros y valores extremos.


In [ ]:
columnas_relacion = ["Distancia", "PASAJEROS"]
for columna in columnas_relacion:
    if not pd.api.types.is_numeric_dtype(df_limpio[columna]):
        raise TypeError(f"{columna} debe tener tipo numérico; tipo actual: {df_limpio[columna].dtype}")

print("Tipos numéricos comprobados:")
print(df_limpio[columnas_relacion].dtypes)
resumen_distancia_pasajeros = df_limpio[columnas_relacion].describe().T.rename(
    columns={
        "count": "Observaciones",
        "mean": "Media",
        "std": "Desviación estándar",
        "min": "Mínimo",
        "25%": "Q1",
        "50%": "Mediana / Q2",
        "75%": "Q3",
        "max": "Máximo"
    }
)
resumen_distancia_pasajeros.index.name = "Variable"
display(resumen_distancia_pasajeros.round(2))

# La muestra se usa únicamente para la visualización, no para las estadísticas anteriores.
tamano_muestra = min(15000, len(df_limpio))
muestra_dispersion = df_limpio[["Distancia", "PASAJEROS"]].sample(
    n=tamano_muestra, random_state=42
)
print(f"Puntos del gráfico: {formato_numero_es(tamano_muestra)} de {formato_numero_es(len(df_limpio))} registros; muestra reproducible (random_state=42).")

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=muestra_dispersion,
    x="Distancia",
    y="PASAJEROS",
    alpha=0.22,
    s=18,
    color="teal",
    edgecolor=None,
    ax=ax
)
ax.set_title("Distancia y pasajeros por registro (muestra aleatoria)")
ax.set_xlabel("Distancia registrada")
ax.set_ylabel("Pasajeros por registro")
fig.tight_layout()
plt.show()


**Hallazgo.** El diagrama de dispersión, construido con una muestra reproducible de **15.000** registros, muestra muchos puntos con pocos pasajeros y una amplia dispersión para distintas distancias. La matriz siguiente cuantifica la asociación lineal en todos los registros.

**Interpretación.** La asociación observada es inversa y débil; el patrón no indica una relación lineal marcada.

**Relación con la pregunta.** La distancia aporta una dimensión cuantitativa para explorar diferencias de pasajeros entre tramos.

**Limitación.** La muestra se usa solo para el gráfico; la dispersión, los extremos y otras variables como ruta y operador limitan lo que puede inferirse. La distancia no explica por sí sola la demanda ni implica causalidad.

### 3.5. Matriz de correlación de variables numéricas

Calcularemos correlación de Pearson para `PASAJEROS` y `Distancia`, las variables cuantitativas directamente vinculadas con la pregunta. No incluimos `Cod_Operador`, porque es un identificador categórico, ni `Año` o `Mes` como si sus códigos fueran magnitudes comparables. La matriz pequeña se presenta como heatmap con valores y escala de color entre −1 y 1.


In [ ]:
variables_correlacion = ["PASAJEROS", "Distancia"]
matriz_correlacion = df_limpio[variables_correlacion].corr(method="pearson")
display(matriz_correlacion.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matriz_correlacion,
    annot=True,
    fmt=".3f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlación de Pearson"},
    ax=ax
)
ax.set_title("Correlación entre pasajeros y distancia")
ax.set_xlabel("Variable")
ax.set_ylabel("Variable")
fig.tight_layout()
plt.show()


**Hallazgo.** La correlación de Pearson entre `Distancia` y `PASAJEROS` es **r = −0,143**.

**Interpretación.** Es una asociación lineal inversa débil. Pearson resume relaciones lineales y puede verse influida por la asimetría y los valores extremos de `PASAJEROS`; no describe por completo una posible relación no lineal.

**Relación con la pregunta.** El resultado aporta evidencia preliminar sobre la asociación entre una característica del tramo y la cantidad de pasajeros registrada.

**Limitación.** La correlación no demuestra causalidad ni permite atribuir a la distancia las diferencias de demanda.

### Síntesis de las relaciones observadas

- Los operadores difieren tanto en pasajeros acumulados como en la distribución por registro; ambas medidas describen aspectos distintos y dependen del volumen y la cobertura de observaciones históricas.
- Las rutas direccionales con mayores acumulados son ANF → SCL, SCL → ANF y BUE → SCL. Estos totales identifican corredores de alto volumen en el período observado, no su comportamiento mensual.
- La distribución nacional muestra una mediana por registro superior a la internacional, mientras que la proporción de ceros es mayor en la categoría internacional.
- La relación lineal entre distancia y pasajeros es débil e inversa (**r = −0,143**); la dispersión y el carácter agregado de los registros limitan su interpretación.

En conjunto, los resultados ayudan a caracterizar cómo varía la demanda según operador, ruta y tipo de tráfico, pero no identifican causas ni tendencias. El análisis de relaciones todavía no muestra cómo evolucionan estos patrones en el tiempo; esa cuestión se abordará en la sección 4 mediante el análisis temporal o contextual.

## 4. Análisis temporal y contextual


La pregunta del proyecto incluye una dimensión temporal: el volumen de pasajeros puede cambiar entre años y meses. En esta sección resumiremos los pasajeros agregados por período y compararemos perfiles mensuales; estos resultados son descriptivos y no explican por sí solos las causas de las variaciones.


### 4.1. Evolución anual de pasajeros

Agruparemos `PASAJEROS` por `Año` para observar cambios en el total anual. Además del cambio absoluto y porcentual respecto del año anterior, comprobaremos la cobertura de años y meses, especialmente la del año más reciente. Los totales anuales no equivalen a pasajeros por registro.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
import plotly.express as px

# Reutilizar el dataset limpio o cargarlo si se ejecuta esta sección por separado.
if "df_limpio" not in globals():
    df_limpio = pd.read_csv("trafico_limpio.csv", sep=";")
    print("Dataset limpio cargado:", df_limpio.shape)
else:
    print("Se reutiliza df_limpio:", df_limpio.shape)

columnas_temporales = ["Año", "Mes", "PASAJEROS"]
for columna in columnas_temporales:
    if not pd.api.types.is_numeric_dtype(df_limpio[columna]):
        raise TypeError(f"{columna} debe ser numérica; tipo actual: {df_limpio[columna].dtype}")
print("Tipos temporales y de la variable principal:")
print(df_limpio[columnas_temporales].dtypes)

meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

def formato_numero_es(valor, decimales=0):
    texto = f"{valor:,.{decimales}f}"
    return texto.replace(",", "_").replace(".", ",").replace("_", ".")


In [ ]:
# Sumar pasajeros por año y calcular variaciones solo entre años consecutivos.
resumen_anual = (
    df_limpio.groupby("Año", dropna=True)["PASAJEROS"]
    .sum()
    .sort_index()
    .rename("Pasajeros anuales")
    .reset_index()
)
resumen_anual["Año anterior"] = resumen_anual["Año"].shift()
resumen_anual["Total año anterior"] = resumen_anual["Pasajeros anuales"].shift()
resumen_anual["Cambio absoluto"] = resumen_anual["Pasajeros anuales"] - resumen_anual["Total año anterior"]
consecutivos = resumen_anual["Año"].diff().eq(1) & resumen_anual["Total año anterior"].ne(0)
resumen_anual["Cambio absoluto"] = resumen_anual["Cambio absoluto"].where(consecutivos)
resumen_anual["Cambio porcentual"] = (
    (resumen_anual["Pasajeros anuales"] / resumen_anual["Total año anterior"] - 1) * 100
).where(consecutivos)

# Mostrar en formato vertical y con separadores legibles.
tabla_anual = resumen_anual[["Año", "Pasajeros anuales", "Cambio absoluto", "Cambio porcentual"]].copy()
tabla_anual["Pasajeros anuales"] = tabla_anual["Pasajeros anuales"].map(formato_numero_es)
tabla_anual["Cambio absoluto"] = tabla_anual["Cambio absoluto"].map(lambda v: "—" if pd.isna(v) else formato_numero_es(v))
tabla_anual["Cambio porcentual"] = tabla_anual["Cambio porcentual"].map(lambda v: "—" if pd.isna(v) else formato_numero_es(v, 2) + " %")
display(tabla_anual)

# Verificar años, pares año-mes observados y meses del año más reciente.
año_minimo = int(df_limpio["Año"].min())
año_maximo = int(df_limpio["Año"].max())
años_esperados = set(range(año_minimo, año_maximo + 1))
años_observados = set(df_limpio["Año"].dropna().astype(int).unique())
años_ausentes = sorted(años_esperados - años_observados)
pares_observados = pd.MultiIndex.from_frame(df_limpio[["Año", "Mes"]].drop_duplicates())
pares_esperados = pd.MultiIndex.from_product(
    [range(año_minimo, año_maximo + 1), range(1, 13)], names=["Año", "Mes"]
)
pares_ausentes = pares_esperados.difference(pares_observados)
meses_ultimo_año = [int(mes) for mes in sorted(df_limpio.loc[df_limpio["Año"].eq(año_maximo), "Mes"].astype(int).unique())]
meses_ultimo_año_ausentes = sorted(set(range(1, 13)) - set(meses_ultimo_año))
print(f"Cobertura observada: {año_minimo}–{año_maximo}; años sin registros: {años_ausentes or 'ninguno'}.")
print(f"Combinaciones año-mes sin registros: {len(pares_ausentes)}.")
print(f"Meses observados en {año_maximo}: {meses_ultimo_año}; meses ausentes: {meses_ultimo_año_ausentes or 'ninguno'}.")

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=resumen_anual, x="Año", y="Pasajeros anuales", marker="o", color="steelblue", ax=ax)
ax.set_title("Evolución anual de pasajeros registrados")
ax.set_xlabel("Año")
ax.set_ylabel("Pasajeros acumulados en el año")
ax.xaxis.set_major_locator(MaxNLocator(nbins=12, integer=True))
ax.yaxis.set_major_formatter(FuncFormatter(lambda valor, posicion: f"{formato_numero_es(valor / 1_000_000, 0)} M"))
fig.tight_layout()
plt.show()


**Hallazgo.** El total anual pasa de **1.110.351** pasajeros en 1984 a **28.386.435** en 2025. La caída interanual más marcada ocurre en **2020** (**-63,95 %**); después se observa una recuperación sostenida y el total de 2024 supera el de 2019. El archivo contiene los doce meses de 2025.

**Interpretación y relación con la pregunta.** La serie muestra un aumento de largo plazo, interrumpido por una caída intensa y seguido por una recuperación. Esto evidencia que el volumen registrado cambia considerablemente entre períodos y aporta contexto para estudiar cómo se distribuye la demanda aérea.

**Limitación.** El CSV confirma presencia de los doce meses, pero no permite certificar por sí solo que los datos de 2025 sean definitivos. Tampoco se atribuyen los cambios a acontecimientos específicos: son totales anuales agregados, no pasajeros por registro ni explicación causal.

### 4.2. Estacionalidad mensual

Sumaremos los pasajeros de cada mes calendario a través de los años. Para reducir la influencia de la cantidad de años, también calcularemos el promedio de los totales mensuales por año para cada mes. La comparación de ambos resúmenes ayuda a explorar una posible estacionalidad, que luego contrastaremos entre años.


In [ ]:
# Calcular primero el total de cada combinación año-mes.
totales_por_anio_mes = (
    df_limpio.groupby(["Año", "Mes"], dropna=True)["PASAJEROS"]
    .sum()
    .rename("Pasajeros del mes")
    .reset_index()
)
resumen_mensual = (
    totales_por_anio_mes.groupby("Mes", dropna=True)["Pasajeros del mes"]
    .agg(Total_periodo="sum", Promedio_anual="mean", Años_observados="count")
    .reindex(range(1, 13))
    .rename_axis("Mes")
    .reset_index()
)
resumen_mensual["Mes"] = resumen_mensual["Mes"].map(dict(enumerate(meses, start=1)))
tabla_mensual = resumen_mensual.copy()
for columna in ["Total_periodo", "Promedio_anual"]:
    tabla_mensual[columna] = tabla_mensual[columna].map(lambda v: "—" if pd.isna(v) else formato_numero_es(v))
display(tabla_mensual.rename(columns={
    "Total_periodo": "Total acumulado",
    "Promedio_anual": "Promedio del total mensual por año",
    "Años_observados": "Años con dato"
}))

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=resumen_mensual, x="Mes", y="Total_periodo", color="seagreen", ax=ax)
ax.set_title("Pasajeros acumulados por mes calendario")
ax.set_xlabel("Mes")
ax.set_ylabel("Pasajeros acumulados en el período")
ax.yaxis.set_major_formatter(FuncFormatter(lambda valor, posicion: f"{formato_numero_es(valor / 1_000_000, 0)} M"))
fig.tight_layout()
plt.show()


**Hallazgo.** Al sumar todo el período, **enero** presenta el mayor total (**42.950.895**) y **junio** el menor (**29.821.758**). El promedio del total mensual por año conserva ese orden y cada mes cuenta con **42 años observados**.

**Interpretación y relación con la pregunta.** La diferencia entre meses sugiere un posible patrón estacional en el volumen registrado. La comparación entre años de la siguiente subsección permite ver si este patrón se repite.

**Limitación.** Los totales acumulados dependen del número de años y de la cobertura de cada período. Aunque están presentes los doce meses en cada año observado, estos agregados no bastan por sí solos para afirmar que la estacionalidad sea estable o recurrente.

### 4.3. Evolución mensual por año

La matriz año × mes permitirá observar diferencias de magnitud y períodos sin registros. Las celdas sin observaciones se conservarán como faltantes y se mostrarán sin color, no como demanda cero. Como los totales absolutos de años distintos pueden diferir mucho, agregaremos un mapa complementario con el porcentaje que cada mes aporta al total observado de su año; este perfil relativo no reemplaza los valores absolutos.


In [ ]:
# Matriz de pasajeros por año y mes; no se rellenan combinaciones ausentes con cero.
años_matriz = range(int(df_limpio["Año"].min()), int(df_limpio["Año"].max()) + 1)
matriz_anio_mes = (
    df_limpio.pivot_table(
        index="Año", columns="Mes", values="PASAJEROS", aggfunc="sum", dropna=True
    )
    .reindex(index=años_matriz, columns=range(1, 13))
)
matriz_anio_mes.columns = meses
matriz_anio_mes.index.name = "Año"

faltantes_anio_mes = matriz_anio_mes.isna().stack()
if faltantes_anio_mes.any():
    periodos_sin_registros = faltantes_anio_mes[faltantes_anio_mes].rename("Sin registros").reset_index()
    display(periodos_sin_registros)
    print(f"Se encontraron {len(periodos_sin_registros)} combinaciones año-mes sin registros; se mantienen como faltantes.")
else:
    print("No hay combinaciones año-mes sin registros dentro del período observado.")

tabla_matriz = matriz_anio_mes.copy().map(
    lambda valor: "—" if pd.isna(valor) else formato_numero_es(valor)
)
display(tabla_matriz)

mapa_colores = plt.get_cmap("YlOrRd").copy()
mapa_colores.set_bad("lightgrey")
fig, ax = plt.subplots(figsize=(13, 14))
sns.heatmap(
    matriz_anio_mes,
    mask=matriz_anio_mes.isna(),
    cmap=mapa_colores,
    linewidths=0.15,
    cbar_kws={"label": "Pasajeros acumulados por año-mes"},
    xticklabels=meses,
    yticklabels=True,
    ax=ax
)
ax.set_title("Pasajeros mensuales por año: valores absolutos")
ax.set_xlabel("Mes")
ax.set_ylabel("Año")
fig.tight_layout()
plt.show()

# Normalizar cada fila para comparar la composición mensual dentro de cada año.
matriz_porcentual = matriz_anio_mes.div(matriz_anio_mes.sum(axis=1, min_count=1), axis=0) * 100
mapa_relativo = plt.get_cmap("YlGnBu").copy()
mapa_relativo.set_bad("lightgrey")
fig, ax = plt.subplots(figsize=(13, 14))
sns.heatmap(
    matriz_porcentual,
    mask=matriz_porcentual.isna(),
    cmap=mapa_relativo,
    vmin=0,
    vmax=matriz_porcentual.max().max(),
    linewidths=0.15,
    cbar_kws={"label": "% del total observado de cada año"},
    xticklabels=meses,
    yticklabels=True,
    ax=ax
)
ax.set_title("Distribución mensual relativa dentro de cada año")
ax.set_xlabel("Mes")
ax.set_ylabel("Año")
fig.tight_layout()
plt.show()


**Hallazgo.** La matriz absoluta confirma 42 años × 12 meses y no presenta combinaciones vacías. En el perfil mensual relativo, **enero** es el mes de mayor participación en **33 de 42 años** y **junio** tiene la menor participación en **28 años**. Las proporciones varían entre años y el perfil de 2020 es especialmente desigual.

**Interpretación y relación con la pregunta.** El mapa absoluto muestra cómo cambia el volumen total y el relativo facilita comparar el calendario interno de cada año, incluso cuando los tamaños anuales difieren. La repetición de enero como máximo y junio como mínimo respalda la existencia de un patrón mensual recurrente en los registros, aunque su intensidad no es idéntica todos los años.

**Limitación.** Las matrices describen asociaciones temporales, no sus causas. La normalización expresa la participación dentro del total observado de cada año, no pasajeros por registro ni crecimiento interanual.

### 4.4. Visualización temporal interactiva

Agregaremos pasajeros por mes y tipo de tráfico (`NAC`) y construiremos una fecha mensual a partir de `Año` y `Mes`. La leyenda de Plotly permitirá activar o desactivar las categorías para comparar sus trayectorias sin dibujar una línea por cada operador. Si falta una combinación mes-categoría, se mantendrá como `NaN` para que la línea muestre un corte, no un cero artificial.


In [ ]:
# Sumar pasajeros y contar registros en cada mes y tipo de tráfico.
mensual_por_trafico = (
    df_limpio.groupby(["Año", "Mes", "NAC"], dropna=True)
    .agg(PASAJEROS=("PASAJEROS", "sum"), Registros=("PASAJEROS", "size"))
    .reset_index()
)
mensual_por_trafico["Fecha"] = pd.to_datetime({
    "year": mensual_por_trafico["Año"].astype(int),
    "month": mensual_por_trafico["Mes"].astype(int),
    "day": 1
})

fecha_inicio = mensual_por_trafico["Fecha"].min()
fecha_fin = mensual_por_trafico["Fecha"].max()
calendario_mensual = pd.date_range(fecha_inicio, fecha_fin, freq="MS")
categorias_trafico = mensual_por_trafico["NAC"].dropna().unique().tolist()
series_completas = []
for categoria in categorias_trafico:
    serie_categoria = (
        mensual_por_trafico.loc[mensual_por_trafico["NAC"].eq(categoria)]
        .set_index("Fecha")
        .reindex(calendario_mensual)
    )
    serie_categoria["NAC"] = categoria
    series_completas.append(serie_categoria.rename_axis("Fecha").reset_index())
serie_interactiva = pd.concat(series_completas, ignore_index=True)

combinaciones_esperadas = len(calendario_mensual) * len(categorias_trafico)
combinaciones_observadas = len(mensual_por_trafico)
print(f"Período mensual: {fecha_inicio:%Y-%m} a {fecha_fin:%Y-%m}.")
print(f"Mes-categoría sin registros: {combinaciones_esperadas - combinaciones_observadas}; se conservan como faltantes, no como cero.")
print("Categorías interactivas (seleccionables desde la leyenda):", categorias_trafico)

fig = px.line(
    serie_interactiva,
    x="Fecha",
    y="PASAJEROS",
    color="NAC",
    custom_data=["Registros"],
    title="Evolución mensual de pasajeros por tipo de tráfico",
    labels={
        "Fecha": "Mes",
        "PASAJEROS": "Pasajeros mensuales agregados",
        "NAC": "Tipo de tráfico"
    }
)
fig.update_traces(
    connectgaps=False,
    hovertemplate=(
        "<b>%{fullData.name}</b><br>Mes: %{x|%b %Y}"
        "<br>Pasajeros agregados: %{y:,.0f}"
        "<br>Registros: %{customdata[0]:,.0f}<extra></extra>"
    )
)
fig.update_layout(
    width=1050,
    height=600,
    hovermode="x unified",
    legend_title_text="Tipo de tráfico",
    margin=dict(t=70, l=70, r=30, b=65)
)
display(fig)

ultimo_año = int(df_limpio["Año"].max())
meses_ultimo_año = [int(mes) for mes in sorted(df_limpio.loc[df_limpio["Año"].eq(ultimo_año), "Mes"].astype(int).unique())]
print(f"El año más reciente ({ultimo_año}) contiene {len(meses_ultimo_año)} meses con registros: {meses_ultimo_año}.")


**Hallazgo.** La serie interactiva presenta el total mensual agregado de tráfico nacional e internacional en líneas separadas; ambas categorías están disponibles para los 504 meses del período observado y el eje termina en diciembre de 2025. La leyenda permite ocultar o mostrar cada trayectoria.

**Interpretación y relación con la pregunta.** La visualización facilita explorar cómo los cambios de largo plazo y las variaciones mensuales difieren entre tipos de tráfico, y complementa la comparación agregada de las subsecciones anteriores.

**Limitación.** El gráfico muestra suma de pasajeros por observación mensual agregada, no pasajeros por vuelo individual. La presencia de los doce meses en 2025 se verifica en el CSV, pero no certifica que los datos de la fuente sean definitivos; tampoco se atribuyen causas a los cambios.

### Síntesis del análisis temporal y contextual

- **Largo plazo:** el total anual aumenta desde **1.110.351** pasajeros en 1984 hasta **28.386.435** en 2025. La mayor caída interanual se registra en 2020 (**-63,95 %**), seguida por una recuperación en los años posteriores; no se atribuyen causas.
- **Patrón mensual posible:** enero reúne el mayor total y junio el menor. En el perfil relativo, enero es el máximo en 33 de 42 años y junio el mínimo en 28, lo que respalda un patrón recurrente cuya intensidad varía entre años.
- **Diferencias entre períodos:** los mapas absolutos y relativos muestran cambios tanto en el volumen anual como en la composición mensual; 2025 cuenta con los doce meses en el archivo.
- **Contexto y límites:** las sumas mensuales y anuales son agregados de registros y no explican causas. La cobertura del archivo no permite confirmar si el último año está cerrado o fue revisado definitivamente.

Estos resultados conectan la pregunta del proyecto con la evolución de la demanda observada por año, mes y tipo de tráfico. La presencia de patrones temporales aporta contexto a las diferencias por operador y ruta, pero no establece por qué ocurren.

## Conclusiones del análisis exploratorio

El EDA aporta evidencia preliminar para responder cómo varía la demanda registrada según ruta, operador y periodo. El archivo original tiene **251.117 filas y 25 columnas**; el conjunto preparado para el análisis principal queda en **251.093 filas y 22 columnas**, sin valores nulos ni duplicados exactos. La preparación excluyó `PAX_LIB`, `CAR_LIB` y `CORREO`, recortó espacios externos, retiró seis filas con información faltante —incluidos registros geográficos de AEROVIAS DAP asociados a `AWC`, cuya corrección no pudo verificarse— y eliminó 18 duplicados exactos. Se conservaron los CSV original y limpio por separado. En los datos originales también se observaron faltantes en `PAX_LIB` y `CORREO`, etiquetas con espacios y magnitudes que requieren contexto; estos hallazgos no bastan para clasificar automáticamente observaciones como erróneas.

La distribución de `PASAJEROS` es asimétrica y dispersa: la media supera ampliamente la mediana y una fracción de observaciones queda sobre el umbral IQR. Estas observaciones son extremos estadísticos, no errores confirmados. Los agregados muestran diferencias relevantes entre operadores y rutas, mientras que la mediana de pasajeros por registro es mayor en el tráfico nacional y la proporción de registros con cero pasajeros es mayor en el internacional. Son medidas complementarias —acumulados, distribuciones por registro y proporciones— y no deben interpretarse como equivalentes ni como evidencia causal.

En el tiempo, el total anual cambia marcadamente, con una caída pronunciada en 2020 y recuperación posterior en los años observados. Enero concentra el mayor total mensual acumulado y junio el menor; además, enero es el mes de mayor participación en 33 de 42 años y junio el menor en 28. La recurrencia sugiere un patrón estacional en los registros, aunque su intensidad varía entre años. El archivo incluye los doce meses de 2025, pero eso no permite confirmar que ese año esté cerrado o que la fuente haya finalizado su revisión.

En conjunto, los resultados respaldan que la demanda observada varía por operador, ruta, tipo de tráfico y mes. Sin embargo, los registros son agregados por ruta, aerolínea, mes y año: no describen vuelos individuales ni explican las causas de las diferencias. La cobertura histórica, las decisiones de limpieza, la falta de verificación de algunos valores extremos y la ausencia de variables externas limitan las inferencias. Para profundizar, se requiere investigar los valores inusuales con la fuente y evaluar con mayor detalle las diferencias temporales y geográficas, sin atribuir causalidad a las asociaciones descriptivas.
